In [ ]:
import sys
sys.path.append("/home/delfina/projects/palace-course/utils")
from verify_topology import verify, print_report
from ihp import PDK
from ihp.cells.inductors import inductor2
from gsim.palace import DrivenSim
from pathlib import Path

PDK.activate()
#yaml_file = Path("~/Documents/IHP/ihp/s_parameters/ihp_stack.yaml").expanduser()
yaml_file = Path("~/projects/IHP/ihp/s_parameters/ihp_stack.yaml").expanduser()

In [ ]:
c = inductor2()

cc = c.copy()
cc.flatten()

c.draw_ports()
c.plot()

In [ ]:
# Checking the port layers.
cc.ports

In [ ]:
sim = DrivenSim()
sim.set_output_dir("runs/palace-sim-inductor2")
sim.set_geometry(cc)

sim.set_stack(substrate_thickness=2.0,air_above=5.0)

sim.add_port("P1",layer="topmetal2",offset=2.0)
sim.add_port("P2",layer="topmetal2",offset=2.0)

sim.set_driven(fmin=1e9, fmax=100e9, num_points=60)

print(sim.validate_config())

In [ ]:
#sim.mesh(preset="default")

In [ ]:
sim.mesh(preset="graded",margin=0.0, refined_mesh_size=1.0)

In [ ]:
sim.plot_mesh(show_groups=["metal","P"],interactive=False)

In [ ]:
sim.plot_mesh(
    style="solid", interactive=False, 
    transparent_groups=["air__None","air__passive","SiO2__None","SiO2__passive","passive__None"],
)

In [ ]:
sim.write_config()
results = sim.run_local()

In [ ]:
result = verify(
    "runs/palace-sim-inductor2/palace.msh",
    "runs/palace-sim-inductor2/config.json"
)
print_report(result)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(results["port-S.csv"])
df.columns = df.columns.str.strip()  # Remove whitespace from column names

freq = df["f (GHz)"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6))

# Magnitude plot
ax1.plot(freq, df["|S[1][1]| (dB)"], marker=".", label="S11")
ax1.plot(freq, df["|S[2][1]| (dB)"], marker=".", label="S21")
ax1.set_xlabel("Frequency (GHz)")
ax1.set_ylabel("Magnitude (dB)")
ax1.set_title("S-Parameters")
ax1.legend()
ax1.grid(True)

# Phase plot
ax2.plot(freq, df["arg(S[1][1]) (deg.)"], marker=".", label="S11")
ax2.plot(freq, df["arg(S[2][1]) (deg.)"], marker=".", label="S21")
ax2.set_xlabel("Frequency (GHz)")
ax2.set_ylabel("Phase (deg)")
ax2.legend()
ax2.grid(True)

plt.tight_layout()